# YOLOv8n Training — SIH26034 Principal Display Panel (PDP) Detection
**Owner:** Shubh (Vision Pipeline)  
**Dataset:** Gauri (~250 augmented images, Roboflow project `legal-metrology-pdp` v1)  
**Hardware:** Google Colab free T4 GPU (`Runtime > Change runtime type > T4 GPU`)

---
### Day 5 Decision Gate (Hard Rule):
- Evaluate `mAP50` upon run completion.
- If `mAP50 >= 0.70`: Export `best.pt` into `backend/app/data/models/yolov8n_pdp.pt` and enable YOLO as primary detector.
- If `mAP50 < 0.70`: Do NOT keep iterating past Day 5. The OpenCV contour detection fallback (`opencv_fallback_detect()`) automatically becomes PRIMARY for the rest of the build.

In [ ]:
# Step 1: Install required packages
!pip install ultralytics roboflow -q

# Verify GPU acceleration
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. Switch to T4 GPU via Runtime > Change runtime type.")

## 2. Authenticate & Download Roboflow Dataset (v1)
**Security Notice:** Do NOT hardcode your API key into git. In Colab, open the **🔑 Secrets** tab on the left sidebar, add a secret named `ROBOFLOW_API_KEY`, and enable notebook access. Alternatively, enter it securely when prompted.

In [ ]:
import os, getpass
from roboflow import Roboflow

# Securely retrieve API Key from Colab Secrets or interactive prompt
try:
    from google.colab import userdata
    api_key = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    api_key = os.environ.get('ROBOFLOW_API_KEY')

if not api_key:
    api_key = getpass.getpass("Enter Roboflow API Key: ")

rf = Roboflow(api_key=api_key)
project = rf.workspace("shubh-v").project("legal-metrology-pdp")
version = project.version(1)
dataset = version.download("yolov8")

print(f"Dataset downloaded to: {dataset.location}")

## 3. Train YOLOv8n (Run #1)
Fine-tuning YOLOv8 nano starting from pretrained weights (`yolov8n.pt`).
- Image Size: 640x640 (standard YOLO input)
- Epochs: 50
- Batch Size: 16

In [ ]:
from ultralytics import YOLO

# Initialize base nano detector
model = YOLO("yolov8n.pt")

# Train on Gauri's dataset
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="sih26034-pdp",
    name="run1",
    patience=15,
    save=True,
    plots=True,
)

## 4. Evaluate Validation Metrics & Day 5 Decision Gate
Check mAP50, precision, and recall on the validation split.

In [ ]:
# Run validation on test/val set
metrics = model.val()
map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)

print("=" * 60)
print(f"Validation Results:")
print(f"  mAP@50:      {map50:.4f}")
print(f"  mAP@50-95:   {map50_95:.4f}")
print("=" * 60)

# Decision gate
if map50 >= 0.70:
    print("STATUS: PASS DECISION GATE.")
    print("mAP50 is high enough to trust for demo. Proceed to download weights.")
else:
    print("STATUS: UNDERPERFORMING DECISION GATE (< 0.70).")
    print("Per project rule, pivot to opencv_fallback_detect() as PRIMARY.")

# Display training loss/metric charts
from IPython.display import Image, display
results_png = "sih26034-pdp/run1/results.png"
if os.path.exists(results_png):
    display(Image(filename=results_png))

## 5. Download Trained Weights
Download `best.pt` to your local machine, and place it at:
`backend/app/data/models/yolov8n_pdp.pt`

In [ ]:
from google.colab import files
best_weights = "sih26034-pdp/run1/weights/best.pt"
if os.path.exists(best_weights):
    print(f"Found best weights at {best_weights}. Triggering browser download...")
    files.download(best_weights)
else:
    print(f"Weights not found at {best_weights}. Check training log for errors.")